In [1]:
from collections.abc import Callable
from typing import Any, ClassVar, Literal

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from clawpack import pyclaw
from context_flux_no.simulations.pdesolve import pdesolve_pyclaw
from jaxtyping import Array, Float, PRNGKeyArray
from tqdm import tqdm


jax.config.update("jax_enable_x64", True)


def riemann_cubic_1D(
    q_l: Float[np.ndarray, "num_eqns num_riemanns"],
    q_r: Float[np.ndarray, "num_eqns num_riemanns"],
    aux_l: Float[np.ndarray, "num_aux num_riemanns"],
    aux_r: Float[np.ndarray, "num_aux num_riemanns"],
    problem_data: dict[str, float],
) -> tuple[
    Float[np.ndarray, "num_eqns num_waves num_riemanns"],
    Float[np.ndarray, "num_waves num_riemanns"],
    Float[np.ndarray, "num_eqns num_riemanns"],
    Float[np.ndarray, "num_eqns num_riemanns"],
]:
    """
    num_riemanns correspond to number of grid cells
    For this problem, num_eqns=1 and num_waves=1.
    """

    a, b, c = problem_data["a"], problem_data["b"], problem_data["c"]

    dq: Float[np.ndarray, "num_eqns num_riemanns"] = q_r - q_l
    # Horner's rule for slightly faster polynomial evaluation
    f_r = ((a * q_r + b) * q_r + c) * q_r
    f_l = ((a * q_l + b) * q_l + c) * q_l
    s_default = (3 * a * q_l + 2 * b) * q_l + c

    s = np.where(np.abs(dq) > 1e-14, (f_r - f_l) / dq, s_default)
    wave = np.expand_dims(dq, axis=1)
    amdq = np.clip(s, max=0.0) * dq
    apdq = np.clip(s, min=0.0) * dq
    return wave, s, amdq, apdq


class CubicFlux1D(eqx.Module):
    n_dim: ClassVar[int] = 1
    n_eqns: ClassVar[int] = 1
    a: float = eqx.field(static=True)
    b: float = eqx.field(static=True)
    c: float = eqx.field(static=True)

    def __init__(self, a: float = 1.0, b: float = 1.0, c: float = 1.0):
        self.a = a
        self.b = b
        self.c = c

    @property
    def coeffs(self) -> dict[str, float]:
        return {"a": self.a, "b": self.b, "c": self.c}

    def solve_pyclaw(
        self,
        ic_factory: Callable[[Float[np.ndarray, " Nx"]], Float[np.ndarray, " Nx"]],
        x_span: tuple[float, float],
        Nx: int,
        t_span: tuple[float, float],
        Nt: int,
        bc: Literal["periodic"],
    ) -> tuple[
        Float[np.ndarray, "time dim x_grid"],
        Float[np.ndarray, " time"],
        Float[np.ndarray, " x_grid"],
    ]:
        solver = pyclaw.ClawSolver1D(riemann_cubic_1D)
        solver.limiters = pyclaw.limiters.tvd.MC
        solver.num_eqn = self.n_eqns
        solver.num_waves = 1
        solver.kernel_language = "Python"
        solver.cfl_desired = 0.5
        solver.cfl_max = 0.9
        solver.fwave = False

        problem_data = {"a": self.a, "b": self.b, "c": self.c}
        return pdesolve_pyclaw(
            solver, problem_data, ic_factory, x_span, Nx, t_span, Nt, bc
        )


In [2]:
def sample_pde(
    pde_factory: Callable[[Any], eqx.Module],
    *,
    seed: int = 0,
    **kwargs: dict[str, tuple[float, float]],
) -> eqx.Module:
    keys = jax.random.split(jax.random.key(seed), len(kwargs))

    param_dict = dict()
    for rng_key, (param_name, sample_range) in zip(keys, kwargs.items()):
        param_dict[param_name] = float(
            jax.random.uniform(
                rng_key,
                minval=sample_range[0],
                maxval=sample_range[1],
            )
        )

    return pde_factory(**param_dict)

In [3]:
pde = sample_pde(CubicFlux1D, a=(-1.0, 1.0), b=(-1.0, 1.0), c=(-1.0, 1.0))
pde

INFO:2025-10-02 21:40:16,440:jax._src.xla_bridge:749: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


2025-10-02 21:40:16,440 INFO CLAW: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


CubicFlux1D(a=0.9398959959797777, b=-0.8387512666713679, c=0.30384451116312405)

In [4]:
from context_flux_no.gaussian_random_field import (
    GaussianCov,
    generate_circulant_embedding_method_1d,
)


u0_func = lambda x_: generate_circulant_embedding_method_1d(
    len(x_), x_[1] - x_[0], GaussianCov(0.05)
)

u, t, x = pde.solve_pyclaw(u0_func, (-1.0, 1.0), 256, (0.0, 0.1), 100, "periodic")

2025-10-02 21:40:32,823 INFO CLAW: Solution 0 computed for time t=0.000000
2025-10-02 21:40:32,828 INFO CLAW: Solution 1 computed for time t=0.001000
2025-10-02 21:40:32,831 INFO CLAW: Solution 2 computed for time t=0.002000
2025-10-02 21:40:32,833 INFO CLAW: Solution 3 computed for time t=0.003000
2025-10-02 21:40:32,836 INFO CLAW: Solution 4 computed for time t=0.004000
2025-10-02 21:40:32,839 INFO CLAW: Solution 5 computed for time t=0.005000
2025-10-02 21:40:32,842 INFO CLAW: Solution 6 computed for time t=0.006000
2025-10-02 21:40:32,844 INFO CLAW: Solution 7 computed for time t=0.007000
2025-10-02 21:40:32,846 INFO CLAW: Solution 8 computed for time t=0.008000
2025-10-02 21:40:32,849 INFO CLAW: Solution 9 computed for time t=0.009000
2025-10-02 21:40:32,851 INFO CLAW: Solution 10 computed for time t=0.010000
2025-10-02 21:40:32,853 INFO CLAW: Solution 11 computed for time t=0.011000
2025-10-02 21:40:32,856 INFO CLAW: Solution 12 computed for time t=0.012000
2025-10-02 21:40:32,85

In [6]:
def sample_coefficients_uniform(
    key: PRNGKeyArray, coeff_range_dict: dict[str, tuple[float, float]]
):
    subkeys = jax.random.split(key, len(coeff_range_dict))
    return {
        name: float(
            jax.random.uniform(subkey, minval=coeff_range[0], maxval=coeff_range[1])
        )
        for subkey, (name, coeff_range) in zip(subkeys, coeff_range_dict.items())
    }


## TODO: Generalize to nd
def solution_to_dataarray(
    u: Float[Array, "time dim x_grid"],
    t: Float[Array, " time"],
    x: Float[Array, " x_grid"],
    coeffs: dict[str, float],
) -> xr.DataArray:
    return xr.DataArray(
        np.expand_dims(u, axis=0),
        coords={
            "t": t,
            "x": x,
            "dim": [
                "u",
            ],
            **{coeff: ("sample", [value]) for coeff, value in coeffs.items()},
        },
        dims=["sample", "t", "dim", "x"],
    )


In [8]:
def generate_dataset(
    n_samples: int,
    pde_factory: Callable[[Any], eqx.Module],
    initial_condition_fn: Callable[[Array, PRNGKeyArray], Array],
    coeff_range_dict: dict,
    x_span: tuple[float, float],
    Nx: int,
    t_span: tuple[float, float],
    Nt: int,
    bc: Literal["periodic"] = "periodic",
    seed: int = 0,
):
    keys = jax.random.split(jax.random.key(seed), n_samples)

    solutions = []
    for key in tqdm(keys):
        key_coeff, key_ic = jax.random.split(key)
        coeffs = sample_coefficients_uniform(key_coeff, coeff_range_dict)
        pde = pde_factory(**coeffs)

        sol = pde.solve_pyclaw(
            lambda u0: initial_condition_fn(u0, key_ic), x_span, Nx, t_span, Nt, bc
        )
        solutions.append(*sol, pde.coeffs)

    return xr.concat(solutions, "sample")

In [4]:
class TruncatedFourier1D(eqx.Module):
    an: Float[Array, " K"]
    bn: Float[Array, " K"]
    a0: Float[Array, ""] = eqx.field(default_factory=lambda: jnp.array(0.0))
    L: float = eqx.field(static=True, default=1.0)

    def __call__(self, x: Float[Array, " *shape"]):
        kx: Float[Array, "*shape K"] = jnp.expand_dims(x, axis=-1) * self.wavenumbers
        cos_kx, sin_kx = jnp.cos(kx), jnp.sin(kx)
        return self.a0 + jnp.mean(
            self.an * cos_kx + self.bn * sin_kx, axis=-1
        ) / jnp.sqrt(2)

    @property
    def num_modes(self) -> int:
        return len(self.an)

    @property
    def wavenumbers(self) -> Float[Array, " K"]:
        return jnp.arange(1, self.num_modes + 1) * 2 * jnp.pi / self.L

    @classmethod
    def with_uniform_rand_coeffs(
        cls,
        num_modes: int,
        L: float = 1.0,
        coeff_range: tuple[float, float] = (1, 1),
        offset_range: tuple[float, float] | None = None,
        *,
        key: PRNGKeyArray = jax.random.PRNGKey(0),
    ):
        key_coeff, key_offset = jax.random.split(key, 2)
        an_bn = jax.random.uniform(
            key_coeff,
            shape=(2, num_modes),
            minval=coeff_range[0],
            maxval=coeff_range[1],
        )
        if offset_range is None:
            a0 = jnp.array(0.0)
        else:
            a0 = jax.random.uniform(
                key_offset, minval=offset_range[0], maxval=offset_range[1]
            )
        return cls(*an_bn, a0, L)

In [ ]:
dataset = generate_dataset(
    n_samples=10,
    pde_factory=CubicFlux1D,
    initial_condition_fn=lambda u0, key: TruncatedFourier1D.with_uniform_rand_coeffs(
        num_modes=4, key=key
    )(u0),
    coeff_range_dict={"a": (-1.0, 1.0), "b": (-1.0, 1.0), "c": (-1.0, 1.0)},
    x_span=(-1, 1),
    Nx=256,
    t_span=(0, 0.1),
    Nt=100,
    seed=0,
)

  0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_4179890/1382419846.py:19: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  pde = pde_factory(**coeffs)


2025-09-17 16:39:55,859 INFO CLAW: Solution 0 computed for time t=0.000000
2025-09-17 16:40:10,560 INFO CLAW: Solution 1 computed for time t=0.001000
2025-09-17 16:40:35,076 INFO CLAW: Solution 2 computed for time t=0.001929
2025-09-17 16:40:58,219 INFO CLAW: Solution 3 computed for time t=0.001929
